# 04 — Tuning the source via parameter scans

Scan a₀ across the ROM model and verify the BGP n_c ∝ γ³ scaling.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from harmonyemissions.config import load_config
from harmonyemissions.scan import run_scan

In [ ]:
base = load_config('../configs/scan_example.yaml')
base = base.model_copy(update={'model': 'rom', 'target': base.target.model_copy(update={'gradient_L_over_lambda': 0.02})})
grid = [{'laser.a0': a0} for a0 in [1, 2, 5, 10, 20, 50]]
points = run_scan(base, grid, output_dir=None, n_jobs=1)

In [ ]:
a0_values = np.array([p.overrides['laser.a0'] for p in points], dtype=float)
n_c = np.array([p.result.diagnostics['n_cutoff'] for p in points])
gamma = np.array([p.result.diagnostics['gamma_mirror_peak'] for p in points])

fig, ax = plt.subplots(figsize=(6, 4))
ax.loglog(gamma, n_c, 'o-', label='ROM output')
ref = gamma**3 * (n_c[0] / gamma[0]**3)
ax.loglog(gamma, ref, '--', label='γ³ reference')
ax.set_xlabel('γ_mirror')
ax.set_ylabel('cutoff harmonic n_c')
ax.legend()
ax.set_title('BGP scaling law: n_c ∝ γ³')
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(6, 4))
for p in points:
    n = p.result.spectrum.coords['harmonic'].values
    s = p.result.spectrum.values
    ax.loglog(n, s / s.max(), label=f'a₀={p.overrides["laser.a0"]}')
ax.set_xlabel('harmonic n')
ax.set_ylabel('normalized spectrum')
ax.legend(fontsize=8)
plt.show()